# Women's Health Equity Index (WHEI) — Methodology Notebook

**Author:** Serge-Alain Nyamsin | [GitHub](https://github.com/sanyamsin) | [Hugging Face](https://huggingface.co/Lokozu)

This notebook documents, step by step, the construction of a composite index measuring **equity in women's health** across 30 African countries. It follows the [OECD/JRC Handbook on Constructing Composite Indicators](https://knowledge4policy.ec.europa.eu/composite-indicators_en) 10-step framework.

**Why a composite index?** Ministries of health and partners (CHAI, UNICEF, WHO) need a single, decomposable signal to prioritize investments but a defensible index must make every methodological choice explicit and test its own robustness. That is what this notebook does.

## 1. Conceptual framework

The WHEI covers three dimensions:

| Dimension | Indicators | Rationale |
|---|---|---|
| **Health outcomes** | Maternal mortality ratio; anemia prevalence (women 15–49) | Final results of the health system |
| **Service coverage** | Skilled birth attendance; ANC 4+ visits; modern contraceptive prevalence | Effective access to care |
| **Social determinants** | Female secondary enrollment | Structural driver of health equity |

Sources: WHO Global Health Observatory, World Bank Open Data, DHS-derived series. The bundled snapshot contains real API values (with reference years per indicator); `src/download_data.py` refreshes it, with a fallback strategy for series that reject the `mrnev` query.

In [1]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import plotly.express as px

from build_index import INDICATOR_COLS, DIRECTIONS, impute, normalize, pca_weights, aggregate, build

raw = pd.read_csv("../data/raw/whei_indicators_snapshot.csv")
raw.head()

,country,iso3,mmr,skilled_birth,anc4,mcpr,female_secondary_enroll,anemia_women
0,Cameroon,CMR,438,69.0,65.0,19.0,52.1,41.0
1,Nigeria,NGA,1047,43.3,57.0,12.0,44.9,55.1
2,Senegal,SEN,261,74.5,57.0,26.0,47.2,53.5
3,Mali,MLI,440,67.3,43.0,16.0,40.1,58.7
4,Niger,NER,441,43.7,38.0,11.0,25.3,49.9


## 2. Data quality & missingness

Before any modelling: how complete is the data?

In [2]:
missing = raw[INDICATOR_COLS].isna().sum().rename("missing_countries").to_frame()
missing["pct"] = (missing["missing_countries"] / len(raw) * 100).round(1)
missing

,missing_countries,pct
mmr,0,0.0
skilled_birth,0,0.0
anc4,0,0.0
mcpr,0,0.0
female_secondary_enroll,2,6.7
anemia_women,0,0.0


Missingness is low (only `female_secondary_enroll` has gaps). We use **median imputation** and crucially keep a per-country flag (`n_imputed`) so users can see which scores rest partly on imputed values. In a production version, multiple imputation (MICE) would be preferred.

In [3]:
imputed, flags = impute(raw)
flags.sum(axis=1).value_counts().rename("countries").to_frame()

,countries
0,28
1,2


## 3. Normalization

Min-max rescaling to [0, 100]. Two indicators (MMR, anemia) are *negative*: higher values mean worse equity, so they are inverted. After this step, **100 always means "best in sample"**.

In [4]:
norm = normalize(imputed)
norm[INDICATOR_COLS].describe().loc[["min", "max", "mean"]].round(1)

,mmr,skilled_birth,anc4,mcpr,female_secondary_enroll,anemia_women
min,0.0,0.0,0.0,0.0,0.0,0.0
max,100.0,100.0,100.0,100.0,100.0,100.0
mean,67.9,66.0,44.8,31.3,42.1,36.2


## 4. Correlation structure

A composite index only makes sense if indicators are related but not redundant.

In [5]:
corr = norm[INDICATOR_COLS].corr()
px.imshow(corr, text_auto=".2f", color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
          title="Correlation between normalized indicators")

## 5. Weighting — three schemes

1. **Equal weights**  the transparent baseline.
2. **PCA weights**  data-driven: loadings of the first principal component.
3. **Expert weights** outcome-focused (MMR weighted 25%), reflecting how a ministry of health might prioritize.

In [6]:
w_pca = pca_weights(norm)
pd.DataFrame({"pca_weight": w_pca}).round(3)

,pca_weight
mmr,0.172
skilled_birth,0.213
anc4,0.159
mcpr,0.188
female_secondary_enroll,0.145
anemia_women,0.123


## 6. Aggregation & results

In [7]:
scores, meta = build(raw)
scores[["country", "whei_equal", "whei_pca", "whei_expert", "rank_equal", "n_imputed"]].head(10)

,country,whei_equal,whei_pca,whei_expert,rank_equal,n_imputed
0,Zimbabwe,78.95,80.64,80.19,1,0
1,Kenya,72.33,73.25,71.20,2,0
2,Rwanda,69.81,70.91,74.32,3,0
3,Zambia,69.41,70.92,74.25,4,0
4,Ghana,65.38,66.10,66.37,5,0
5,Malawi,65.18,67.85,68.83,6,0
6,Congo Republic,59.10,61.89,63.60,7,0
7,Gabon,55.61,59.52,61.64,8,1
8,Uganda,55.51,56.83,61.01,9,0
9,Sierra Leone,55.16,57.48,58.55,10,0


In [8]:
px.bar(scores.sort_values("whei_equal"), x="whei_equal", y="country", orientation="h",
       color="whei_equal", color_continuous_scale="RdYlGn", height=700,
       title="WHEI — equal weights", labels={"whei_equal": "score", "country": ""})

## 7. Sensitivity analysis

**The key robustness question:** do country rankings depend on the weighting choice? We compute Spearman rank correlations between the three schemes.

In [9]:
meta["rank_correlation"]

{'equal_vs_pca': 0.988, 'equal_vs_expert': 0.986, 'pca_vs_expert': 0.984}

Rank correlations above 0.98 across all pairs: **the ranking is robust to the weighting scheme**, which is the single most common criticism of composite indices. Countries that move the most between schemes are those with unbalanced profiles (strong coverage, weak outcomes, or vice versa).

In [10]:
scores["rank_spread"] = scores[["rank_equal", "rank_pca", "rank_expert"]].max(axis=1) - scores[["rank_equal", "rank_pca", "rank_expert"]].min(axis=1)
scores.nlargest(5, "rank_spread")[["country", "rank_equal", "rank_pca", "rank_expert", "rank_spread"]]

,country,rank_equal,rank_pca,rank_expert,rank_spread
12,Ethiopia,13,18,15,5
15,Mozambique,16,14,11,5
11,Cameroon,12,12,16,4
14,Tanzania,15,16,13,3
17,Burkina Faso,18,15,17,3


## 8. Limitations & next steps

- **Temporal comparability**: indicators use the latest available year per country, which differs across sources.
- **National averages hide inequity**: the natural extension is a *sub-national* WHEI using DHS regional estimates and wealth-quintile decomposition — precisely where equity gaps live.
- **Normalization is sample-relative**: adding/removing countries shifts scores. Goalpost normalization (fixed benchmarks, as in the HDI) would fix this.
- **Uncertainty**: a Monte Carlo simulation over imputed values and weights would produce confidence intervals on ranks.

---
*Pipeline code: [`src/build_index.py`](../src/build_index.py) · Interactive app: [`app/streamlit_app.py`](../app/streamlit_app.py) · Tests: [`tests/`](../tests/)*